# DETECCIÓN DE PROYECTOS DE INVERSIÓN PÚBLICA CON EJECUCIÓN PRESUPUESTAL ATÍPICA

**1. PROBLEMA:**
Buscamos identificar proyectos de inversión pública cuyo patrón de ejecución presupuestal se aleje significativamente del comportamiento típico. El objetivo es priorizar la atención hacia proyectos con riesgo de subejecución, sobreejecución inusual o posible paralización (brecha entre avance financiero y físico).

**2. Fuentes de datos a integrar:** 
*   **Fuente 1 (MEF):** Seguimiento de Proyectos de Inversión (Datos financieros: PIA, PIM, Devengado). https://fs.datosabiertos.mef.gob.pe/datastorefiles/2026-Seguimiento-PI.csv
*   **Fuente 2 (MEF):** Avance Físico de Obras (Datos físicos: % Avance, Etapa, Valorización). https://fs.datosabiertos.mef.gob.pe/datastorefiles/PROCESO_SELECCION.csv

**3. Unidad de análisis:**
Proyecto de inversión pública dentro de una entidad ejecutora, por año fiscal.

**4. Llave de integración (Columnas comunes):**
Ambas bases no comparten nombres de columnas idénticos, pero semánticamente se integran a través del **Código Único de Inversión (CUI)**:
*   En MEF: `PRODUCTO_PROYECTO`
*   En MEF_2: `CODIGO_UNICO`

<a name='manejoarchivos'></a>
# <font color=blue> INSPECCIÓN INICIAL E INTEGRACIÓN DE FUENTES</font>

### <font color=green> 1. Importar datos </font>

In [ ]:
# Librería principal para trabajar con datos en tablas
import pandas as pd

# Para que Pandas muestre más columnas en pantalla
pd.set_option("display.max_columns", 100)

# ============================================================
#  LEER LAS FUENTES DE DATOS
# ============================================================

# Asegúrate de subir tus archivos reales a Colab con estos nombres
df_mef = pd.read_csv("Seguimiento-PI.csv", encoding='utf-8-sig')
df_mef_2 = pd.read_csv("Proceso_Seleccion.csv", encoding='utf-8-sig')

print("Fuente 1 (MEF - Financiero):")
display(df_mef.head())

print("Fuente 2 (mef_2 - Avance Físico):")
display(df_mef_2.head())

C:\Users\ferna\AppData\Local\Temp\ipykernel_18120\1850918653.py:12: DtypeWarning: Columns (0: SECTOR, 1: PLIEGO) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mef = pd.read_csv("2026-Seguimiento-PI.csv", encoding='utf-8-sig')
C:\Users\ferna\AppData\Local\Temp\ipykernel_18120\1850918653.py:13: DtypeWarning: Columns (0: VAL_META_CAPAC, 1: DES_OBSERVACIONES) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mef_2 = pd.read_csv("Proceso_Seleccion.csv", encoding='utf-8-sig')


Fuente 1 (MEF - Financiero):


,ANO_EJE,NIVEL_GOBIERNO,NIVEL_GOBIERNO_NOMBRE,SECTOR,SECTOR_NOMBRE,PLIEGO,PLIEGO_NOMBRE,SEC_EJEC,EJECUTORA,EJECUTORA_NOMBRE,DEPARTAMENTO_EJECUTORA,DEPARTAMENTO_EJECUTORA_NOMBRE,PROVINCIA_EJECUTORA,PROVINCIA_EJECUTORA_NOMBRE,DISTRITO_EJECUTORA,DISTRITO_EJECUTORA_NOMBRE,TIPO_ACT_PROY,TIPO_ACT_PROY_NOMBRE,PRODUCTO_PROYECTO,PRODUCTO_PROYECTO_NOMBRE,FUNCION,FUNCION_NOMBRE,DEPARTAMENTO_META,DEPARTAMENTO_META_NOMBRE,FUENTE_FINANCIAMIENTO,FUENTE_FINANCIAMIENTO_NOMBRE,RUBRO,RUBRO_NOMBRE,COSTO_ACTUAL,MONTO_EJECUCION_HASTA_HACE_2_ANOS,MONTO_EJECUCION_ANO_ANTERIOR,MONTO_PIA,MONTO_PIM,MONTO_DEVENGADO_ANO_EJE,MONTO_EJECUCION_TOTAL
0,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,460,GOBIERNO REGIONAL TACNA,931,1,REGION TACNA-SEDE CENTRAL,23,TACNA,1,TACNA,1,REGION TACNA-SEDE CENTRAL,2,PROYECTO,2015918,PROYECTO VIAL TACNA LA PAZ,16,TRANSPORTE,23,TACNA,1,RECURSOS ORDINARIOS,15,FONDO DE COMPENSACION REGIONAL,0.0,97546.66,0.0,0,0,0.0,97546.66
1,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,460,GOBIERNO REGIONAL TACNA,1210,2,REGION TACNA - PROY. ESP. RECURSOS HIDRICOS TACNA,0,,0,,0,,2,PROYECTO,2000351,OPERACION Y MANTENIMIENTO,4,AGRARIA,23,TACNA,2,RECURSOS DIRECTAMENTE RECAUDADOS,9,RECURSOS DIRECTAMENTE RECAUDADOS,0.0,1617683.93,0.0,0,0,0.0,1617683.93
2,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,448,GOBIERNO REGIONAL HUANUCO,804,1,REGION HUANUCO-SEDE CENTRAL,10,HUANUCO,1,HUANUCO,2,REGION HUANUCO-SEDE CENTRAL,2,PROYECTO,2001707,LIQUIDACION DE OBRAS,9,EDUCACION Y CULTURA,10,HUANUCO,1,RECURSOS ORDINARIOS,15,FONDO DE COMPENSACION REGIONAL,0.0,174390.21,0.0,0,0,0.0,174390.21
3,2006,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,446,GOBIERNO REGIONAL CUSCO,791,3,REGION CUSCO-PROYECTO ESPECIAL PLAN MERISS,8,CUSCO,1,CUSCO,8,REGION CUSCO-PROYECTO ESPECIAL PLAN MERISS,2,PROYECTO,2001621,ESTUDIOS DE PRE-INVERSION,14,SALUD Y SANEAMIENTO,8,CUSCO,5,RECURSOS DETERMINADOS,1,"CANON, SOBRECANON, REGALIAS Y PARTICIPACIONES",0.0,93332.61,0.0,0,0,0.0,93332.61
4,2006,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,461,GOBIERNO REGIONAL TUMBES,939,300,REGION TUMBES-EDUCACION,24,TUMBES,1,TUMBES,1,REGION TUMBES-EDUCACION,2,PROYECTO,2000270,GESTION DE PROYECTOS,9,EDUCACION Y CULTURA,24,TUMBES,5,RECURSOS DETERMINADOS,1,"CANON, SOBRECANON, REGALIAS Y PARTICIPACIONES",0.0,46500.00,0.0,0,0,0.0,46500.00


Fuente 2 (mef_2 - Avance Físico):


,CODIGO_UNICO,DES_PRODUCTO,DES_ACCION,DES_TIPO_COMPONENTE,DES_UM_PRODU,VAL_META_PRODU,DES_UM_CAPAC,VAL_META_CAPAC,COSTO_INVERSION,PERIODO,VALORIZ_ACUM,AVANCE,DES_OBSERVACIONES,DES_ETAPA
0,2318999,Producto 1: LABORES MINERAS SUBTERRANEAS TRATA...,Cierre con Tapón hermético Bocaminas,INFRAESTRUCTURA,ESPACIOS FISICOS,43.28,M3,"43,28",455417.95,2019-08,455417.95,NaN,NaN,CONTRACTUAL
1,2318999,Producto 1: LABORES MINERAS SUBTERRANEAS TRATA...,Cierre con Tapón hermético Bocaminas,INFRAESTRUCTURA,ESPACIOS FISICOS,43.28,M3,"43,28",455417.95,IV-2017,331867.00,NaN,NaN,EJECUCION
2,2318999,Producto 1: LABORES MINERAS SUBTERRANEAS TRATA...,Cierre con Tapón hermético Bocaminas,INFRAESTRUCTURA,ESPACIOS FISICOS,43.28,M3,"43,28",455417.95,IV-2017,503436.58,NaN,NaN,CRONOGRAMA
3,2318999,Producto 1: LABORES MINERAS SUBTERRANEAS TRATA...,Cierre con Tapón hermético Bocaminas,INFRAESTRUCTURA,ESPACIOS FISICOS,43.28,M3,"43,28",455417.95,I-2018,427276.00,NaN,NaN,EJECUCION
4,2318999,Producto 1: LABORES MINERAS SUBTERRANEAS TRATA...,Cerramiento con concreto simple media barreta,INFRAESTRUCTURA,ESPACIOS FISICOS,9.20,M3,"9,2",114969.32,2019-08,114969.32,NaN,NaN,CONTRACTUAL


### <font color=green> 2. REVISAR EL TAMAÑO DE CADA FUENTE </font>

In [6]:
# shape devuelve: (número de filas, número de columnas)

print("Tamaño de fuente 1 (MEF):", df_mef.shape)
print("Tamaño de fuente 2 (mef_2):", df_mef_2.shape)

Tamaño de fuente 1 (MEF): (696688, 35)
Tamaño de fuente 2 (mef_2): (5672274, 14)


### <font color=green> 3. REVISAR LOS TIPOS DE DATOS DETECTADOS </font>

###### Ejemplo: `PRODUCTO_PROYECTO` o `CODIGO_UNICO` son identificadores, no montos a sumar.


In [8]:
print("Tipos de datos en fuente 1 (MEF):")
display(df_mef.dtypes)

print("Tipos de datos en fuente 2 (MEF_2):")
display(df_mef_2.dtypes)

Tipos de datos en fuente 1 (MEF):


ANO_EJE                                int64
NIVEL_GOBIERNO                           str
NIVEL_GOBIERNO_NOMBRE                    str
SECTOR                                object
SECTOR_NOMBRE                            str
PLIEGO                                object
PLIEGO_NOMBRE                            str
SEC_EJEC                               int64
EJECUTORA                              int64
EJECUTORA_NOMBRE                         str
DEPARTAMENTO_EJECUTORA                 int64
DEPARTAMENTO_EJECUTORA_NOMBRE            str
PROVINCIA_EJECUTORA                    int64
PROVINCIA_EJECUTORA_NOMBRE               str
DISTRITO_EJECUTORA                     int64
DISTRITO_EJECUTORA_NOMBRE                str
TIPO_ACT_PROY                          int64
TIPO_ACT_PROY_NOMBRE                     str
PRODUCTO_PROYECTO                      int64
PRODUCTO_PROYECTO_NOMBRE                 str
FUNCION                                int64
FUNCION_NOMBRE                           str
DEPARTAMEN

Tipos de datos en fuente 2 (MEF_2):


CODIGO_UNICO             int64
DES_PRODUCTO               str
DES_ACCION                 str
DES_TIPO_COMPONENTE        str
DES_UM_PRODU               str
VAL_META_PRODU         float64
DES_UM_CAPAC               str
VAL_META_CAPAC          object
COSTO_INVERSION        float64
PERIODO                    str
VALORIZ_ACUM           float64
AVANCE                 float64
DES_OBSERVACIONES          str
DES_ETAPA                  str
dtype: object

### <font color=green> 4. USAR describe() PARA RESUMIR CADA FUENTE </font>

In [9]:
# describe(include="all") resume variables numéricas y categóricas.

print("Resumen de fuente 1 (MEF):")
display(df_mef.describe(include="all").T)

print("Resumen de fuente 2 (mef_2):")
display(df_mef_2.describe(include="all").T)

Resumen de fuente 1 (MEF):


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ANO_EJE,696688.0,NaN,NaN,NaN,2016.487166,6.624322,2005.0,2010.0,2016.0,2023.0,2026.0
NIVEL_GOBIERNO,696688,3,M,616784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NIVEL_GOBIERNO_NOMBRE,696688,3,GOBIERNOS LOCALES,616784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SECTOR,696688,61,,616784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SECTOR_NOMBRE,696688,33,,616784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLIEGO,696688,331,,616784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLIEGO_NOMBRE,696688,233,,616784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SEC_EJEC,696688.0,NaN,NaN,NaN,266589.69662,95545.246686,3.0,300316.0,300794.0,301327.0,360001.0
EJECUTORA,696688.0,NaN,NaN,NaN,95023.394369,70591.29579,1.0,30607.0,81211.0,150403.0,250401.0
EJECUTORA_NOMBRE,696688,2520,PROGRAMA NACIONAL DE SANEAMIENTO RURAL,5620,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Resumen de fuente 2 (mef_2):


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CODIGO_UNICO,5672274.0,NaN,NaN,NaN,2422010.484641,157529.433842,2000680.0,2304610.0,2446389.0,2536658.0,2738211.0
DES_PRODUCTO,5576200,211122,INFRAESTRUCTURA,452333,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DES_ACCION,5086571,202193,Inventario físico Covid - 19,456942,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DES_TIPO_COMPONENTE,5181599,9,INFRAESTRUCTURA,3674595,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DES_UM_PRODU,4208062,536,NÚMERO DE ESTRUCTURAS FÍSICAS,1176568,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VAL_META_PRODU,4494730.0,NaN,NaN,NaN,75564.631257,56769980.443267,0.0,1.0,1.0,5.0,100000000000.0
DES_UM_CAPAC,2881268,66,M2,1581437,NaN,NaN,NaN,NaN,NaN,NaN,NaN
VAL_META_CAPAC,4222177.0,269654.0,1.0,610374.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COSTO_INVERSION,5671764.0,NaN,NaN,NaN,1904322.731311,46571343.383728,-10412591.87,7820.02,78687.935,441156.465,24909061905.369999
PERIODO,4253898,2019,2021-01,68927,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### <font color=green> 5. REVISAR VALORES FALTANTES </font>

In [10]:
print("Valores faltantes en fuente 1 (MEF):")
display(df_mef.isnull().sum())

print("Valores faltantes en fuente 2 (mef_2):")
display(df_mef_2.isnull().sum())

# También podemos ver el porcentaje de valores faltantes.
print("Porcentaje de valores faltantes en fuente 1 (MEF):")
display((df_mef.isnull().mean() * 100).round(2))

print("Porcentaje de valores faltantes en fuente 2 (mef_2):")
display((df_mef_2.isnull().mean() * 100).round(2))

Valores faltantes en fuente 1 (MEF):


ANO_EJE                              0
NIVEL_GOBIERNO                       0
NIVEL_GOBIERNO_NOMBRE                0
SECTOR                               0
SECTOR_NOMBRE                        0
PLIEGO                               0
PLIEGO_NOMBRE                        0
SEC_EJEC                             0
EJECUTORA                            0
EJECUTORA_NOMBRE                     0
DEPARTAMENTO_EJECUTORA               0
DEPARTAMENTO_EJECUTORA_NOMBRE        0
PROVINCIA_EJECUTORA                  0
PROVINCIA_EJECUTORA_NOMBRE           0
DISTRITO_EJECUTORA                   0
DISTRITO_EJECUTORA_NOMBRE            0
TIPO_ACT_PROY                        0
TIPO_ACT_PROY_NOMBRE                 0
PRODUCTO_PROYECTO                    0
PRODUCTO_PROYECTO_NOMBRE             0
FUNCION                              0
FUNCION_NOMBRE                       0
DEPARTAMENTO_META                    0
DEPARTAMENTO_META_NOMBRE             0
FUENTE_FINANCIAMIENTO                0
FUENTE_FINANCIAMIENTO_NOM

Valores faltantes en fuente 2 (mef_2):


CODIGO_UNICO                 0
DES_PRODUCTO             96074
DES_ACCION              585703
DES_TIPO_COMPONENTE     490675
DES_UM_PRODU           1464212
VAL_META_PRODU         1177544
DES_UM_CAPAC           2791006
VAL_META_CAPAC         1450097
COSTO_INVERSION            510
PERIODO                1418376
VALORIZ_ACUM           4334993
AVANCE                 5078775
DES_OBSERVACIONES      3344533
DES_ETAPA                    0
dtype: int64

Porcentaje de valores faltantes en fuente 1 (MEF):


ANO_EJE                              0.0
NIVEL_GOBIERNO                       0.0
NIVEL_GOBIERNO_NOMBRE                0.0
SECTOR                               0.0
SECTOR_NOMBRE                        0.0
PLIEGO                               0.0
PLIEGO_NOMBRE                        0.0
SEC_EJEC                             0.0
EJECUTORA                            0.0
EJECUTORA_NOMBRE                     0.0
DEPARTAMENTO_EJECUTORA               0.0
DEPARTAMENTO_EJECUTORA_NOMBRE        0.0
PROVINCIA_EJECUTORA                  0.0
PROVINCIA_EJECUTORA_NOMBRE           0.0
DISTRITO_EJECUTORA                   0.0
DISTRITO_EJECUTORA_NOMBRE            0.0
TIPO_ACT_PROY                        0.0
TIPO_ACT_PROY_NOMBRE                 0.0
PRODUCTO_PROYECTO                    0.0
PRODUCTO_PROYECTO_NOMBRE             0.0
FUNCION                              0.0
FUNCION_NOMBRE                       0.0
DEPARTAMENTO_META                    0.0
DEPARTAMENTO_META_NOMBRE             0.0
FUENTE_FINANCIAM

Porcentaje de valores faltantes en fuente 2 (mef_2):


CODIGO_UNICO            0.00
DES_PRODUCTO            1.69
DES_ACCION             10.33
DES_TIPO_COMPONENTE     8.65
DES_UM_PRODU           25.81
VAL_META_PRODU         20.76
DES_UM_CAPAC           49.20
VAL_META_CAPAC         25.56
COSTO_INVERSION         0.01
PERIODO                25.01
VALORIZ_ACUM           76.42
AVANCE                 89.54
DES_OBSERVACIONES      58.96
DES_ETAPA               0.00
dtype: float64

### <font color=green> 6. REVISAR CANTIDAD DE VALORES DISTINTOS POR VARIABLE </font>

In [11]:
# Esto ayuda a detectar variables categóricas, identificadores
# o categorías sospechosas.

print("Valores distintos por variable en fuente 1 (MEF):")
display(df_mef.nunique())

print("Valores distintos por variable en fuente 2 (mef_2):")
display(df_mef_2.nunique())

Valores distintos por variable en fuente 1 (MEF):


ANO_EJE                                  22
NIVEL_GOBIERNO                            3
NIVEL_GOBIERNO_NOMBRE                     3
SECTOR                                   61
SECTOR_NOMBRE                            33
PLIEGO                                  331
PLIEGO_NOMBRE                           233
SEC_EJEC                               2570
EJECUTORA                              2009
EJECUTORA_NOMBRE                       2520
DEPARTAMENTO_EJECUTORA                   26
DEPARTAMENTO_EJECUTORA_NOMBRE            26
PROVINCIA_EJECUTORA                      21
PROVINCIA_EJECUTORA_NOMBRE              197
DISTRITO_EJECUTORA                       44
DISTRITO_EJECUTORA_NOMBRE              2521
TIPO_ACT_PROY                             1
TIPO_ACT_PROY_NOMBRE                      1
PRODUCTO_PROYECTO                     52677
PRODUCTO_PROYECTO_NOMBRE              53314
FUNCION                                  24
FUNCION_NOMBRE                           33
DEPARTAMENTO_META               

Valores distintos por variable en fuente 2 (mef_2):


CODIGO_UNICO            204162
DES_PRODUCTO            211122
DES_ACCION              202193
DES_TIPO_COMPONENTE          9
DES_UM_PRODU               536
VAL_META_PRODU           39462
DES_UM_CAPAC                66
VAL_META_CAPAC          269654
COSTO_INVERSION        1834777
PERIODO                   2019
VALORIZ_ACUM            621446
AVANCE                   12157
DES_OBSERVACIONES       558705
DES_ETAPA                    7
dtype: int64

### <font color=green> 7. REVISAR LOS VALORES DE UNA VARIABLE CATEGÓRICA </font>

In [12]:
# En el MEF revisamos el Nivel de Gobierno y en mef_2 la Etapa del proyecto.

print("Valores de la variable NIVEL_GOBIERNO_NOMBRE en MEF:")
display(df_mef["NIVEL_GOBIERNO_NOMBRE"].value_counts(dropna=False))

print("Valores de la variable DES_ETAPA en mef_2:")
display(df_mef_2["DES_ETAPA"].value_counts(dropna=False))

Valores de la variable NIVEL_GOBIERNO_NOMBRE en MEF:


NIVEL_GOBIERNO_NOMBRE
GOBIERNOS LOCALES       616784
GOBIERNOS REGIONALES     42349
GOBIERNO NACIONAL        37555
Name: count, dtype: int64

Valores de la variable DES_ETAPA en mef_2:


DES_ETAPA
CONSISTENCIA              1193602
Ejecución física (C)      1158941
Expediente técnico (B)    1001091
FYE                        981359
EJECUCION                  648548
CRONOGRAMA                 375734
CONTRACTUAL                312999
Name: count, dtype: int64

### <font color=green> 8. CREAR UNA TABLA RESUMEN SIMPLE PARA CADA FUENTE </font>

In [13]:
resumen_mef = pd.DataFrame({
    "fuente": "mef_seguimiento_financiero",
    "variable": df_mef.columns,
    "tipo_python": df_mef.dtypes.values,
    "n_faltantes": df_mef.isnull().sum().values,
    "porcentaje_faltantes": (df_mef.isnull().mean() * 100).round(2).values,
    "valores_distintos": df_mef.nunique().values
})

resumen_mef_2 = pd.DataFrame({
    "fuente": "mef_2_avance_fisico",
    "variable": df_mef_2.columns,
    "tipo_python": df_mef_2.dtypes.values,
    "n_faltantes": df_mef_2.isnull().sum().values,
    "porcentaje_faltantes": (df_mef_2.isnull().mean() * 100).round(2).values,
    "valores_distintos": df_mef_2.nunique().values
})

tabla_resumen_fuentes = pd.concat([resumen_mef, resumen_mef_2], ignore_index=True)
#Nota: ignore_index=True hace que Pandas cree un nuevo índice consecutivo al unir las tablas.

print("Tabla resumen de las fuentes:")
display(tabla_resumen_fuentes)

Tabla resumen de las fuentes:


,fuente,variable,tipo_python,n_faltantes,porcentaje_faltantes,valores_distintos
0,mef_seguimiento_financiero,ANO_EJE,int64,0,0.00,22
1,mef_seguimiento_financiero,NIVEL_GOBIERNO,str,0,0.00,3
2,mef_seguimiento_financiero,NIVEL_GOBIERNO_NOMBRE,str,0,0.00,3
3,mef_seguimiento_financiero,SECTOR,object,0,0.00,61
4,mef_seguimiento_financiero,SECTOR_NOMBRE,str,0,0.00,33
5,mef_seguimiento_financiero,PLIEGO,object,0,0.00,331
6,mef_seguimiento_financiero,PLIEGO_NOMBRE,str,0,0.00,233
7,mef_seguimiento_financiero,SEC_EJEC,int64,0,0.00,2570
8,mef_seguimiento_financiero,EJECUTORA,int64,0,0.00,2009
9,mef_seguimiento_financiero,EJECUTORA_NOMBRE,str,0,0.00,2520


### <font color=green> 9. IDENTIFICAR COLUMNAS COMUNES ENTRE LAS FUENTES </font>
**Nota del equipo:** Aunque el nombre de las variables difiera entre las entidades (MEF y mef_2), nuestro análisis de los diccionarios determinó que la columna `PRODUCTO_PROYECTO` del MEF corresponde funcionalmente a la columna `CODIGO_UNICO` de mef_2. Esta será nuestra llave de integración.

In [14]:
columnas_comunes = set(df_mef.columns).intersection(set(df_mef_2.columns))

print("Columnas con el mismo nombre exacto entre las fuentes:")
print(columnas_comunes)
print("\nLa integración se forzará mapeando PRODUCTO_PROYECTO con CODIGO_UNICO.")

Columnas con el mismo nombre exacto entre las fuentes:
set()

La integración se forzará mapeando PRODUCTO_PROYECTO con CODIGO_UNICO.


### <font color=green> 10. INTEGRAR LAS DOS FUENTES </font>

In [15]:
# Usamos un Left Join (how="left") para mantener todos los registros del MEF,
# e incorporar el avance físico del mef_2 donde haga match.

base_integrada = pd.merge(
    df_mef,
    df_mef_2,
    left_on="PRODUCTO_PROYECTO",
    right_on="CODIGO_UNICO",
    how="left",
    suffixes=("_mef", "_mef_2"),
    indicator=True
)

print("Base integrada:")
display(base_integrada.head())

Base integrada:


,ANO_EJE,NIVEL_GOBIERNO,NIVEL_GOBIERNO_NOMBRE,SECTOR,SECTOR_NOMBRE,PLIEGO,PLIEGO_NOMBRE,SEC_EJEC,EJECUTORA,EJECUTORA_NOMBRE,DEPARTAMENTO_EJECUTORA,DEPARTAMENTO_EJECUTORA_NOMBRE,PROVINCIA_EJECUTORA,PROVINCIA_EJECUTORA_NOMBRE,DISTRITO_EJECUTORA,DISTRITO_EJECUTORA_NOMBRE,TIPO_ACT_PROY,TIPO_ACT_PROY_NOMBRE,PRODUCTO_PROYECTO,PRODUCTO_PROYECTO_NOMBRE,FUNCION,FUNCION_NOMBRE,DEPARTAMENTO_META,DEPARTAMENTO_META_NOMBRE,FUENTE_FINANCIAMIENTO,FUENTE_FINANCIAMIENTO_NOMBRE,RUBRO,RUBRO_NOMBRE,COSTO_ACTUAL,MONTO_EJECUCION_HASTA_HACE_2_ANOS,MONTO_EJECUCION_ANO_ANTERIOR,MONTO_PIA,MONTO_PIM,MONTO_DEVENGADO_ANO_EJE,MONTO_EJECUCION_TOTAL,CODIGO_UNICO,DES_PRODUCTO,DES_ACCION,DES_TIPO_COMPONENTE,DES_UM_PRODU,VAL_META_PRODU,DES_UM_CAPAC,VAL_META_CAPAC,COSTO_INVERSION,PERIODO,VALORIZ_ACUM,AVANCE,DES_OBSERVACIONES,DES_ETAPA,_merge
0,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,460,GOBIERNO REGIONAL TACNA,931,1,REGION TACNA-SEDE CENTRAL,23,TACNA,1,TACNA,1,REGION TACNA-SEDE CENTRAL,2,PROYECTO,2015918,PROYECTO VIAL TACNA LA PAZ,16,TRANSPORTE,23,TACNA,1,RECURSOS ORDINARIOS,15,FONDO DE COMPENSACION REGIONAL,0.0,97546.66,0.0,0,0,0.0,97546.66,2015918.0,"RED VIAL INTEGRACION VIAL TACNA- LA PAZ, TRAMO...",Tramo I: km 43+610- km 94+000,INFRAESTRUCTURA,ESTRUCTURAS FISICAS,52.3,KM,"52,3",2.934237e+08,2019-07,2.600596e+08,95.99,NaN,EJECUCION,both
1,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,460,GOBIERNO REGIONAL TACNA,931,1,REGION TACNA-SEDE CENTRAL,23,TACNA,1,TACNA,1,REGION TACNA-SEDE CENTRAL,2,PROYECTO,2015918,PROYECTO VIAL TACNA LA PAZ,16,TRANSPORTE,23,TACNA,1,RECURSOS ORDINARIOS,15,FONDO DE COMPENSACION REGIONAL,0.0,97546.66,0.0,0,0,0.0,97546.66,2015918.0,"RED VIAL INTEGRACION VIAL TACNA- LA PAZ, TRAMO...",Tramo I: km 43+610- km 94+000,INFRAESTRUCTURA,ESTRUCTURAS FISICAS,52.3,KM,"52,3",2.934237e+08,2019-05,2.600596e+08,95.99,NaN,EJECUCION,both
2,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,460,GOBIERNO REGIONAL TACNA,931,1,REGION TACNA-SEDE CENTRAL,23,TACNA,1,TACNA,1,REGION TACNA-SEDE CENTRAL,2,PROYECTO,2015918,PROYECTO VIAL TACNA LA PAZ,16,TRANSPORTE,23,TACNA,1,RECURSOS ORDINARIOS,15,FONDO DE COMPENSACION REGIONAL,0.0,97546.66,0.0,0,0,0.0,97546.66,2015918.0,"RED VIAL INTEGRACION VIAL TACNA- LA PAZ, TRAMO...",Tramo I: km 43+610- km 94+000,INFRAESTRUCTURA,ESTRUCTURAS FISICAS,52.3,KM,"52,3",2.934237e+08,2019-04,2.599008e+08,NaN,NaN,EJECUCION,both
3,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,460,GOBIERNO REGIONAL TACNA,931,1,REGION TACNA-SEDE CENTRAL,23,TACNA,1,TACNA,1,REGION TACNA-SEDE CENTRAL,2,PROYECTO,2015918,PROYECTO VIAL TACNA LA PAZ,16,TRANSPORTE,23,TACNA,1,RECURSOS ORDINARIOS,15,FONDO DE COMPENSACION REGIONAL,0.0,97546.66,0.0,0,0,0.0,97546.66,2015918.0,"RED VIAL INTEGRACION VIAL TACNA- LA PAZ, TRAMO...",Tramo I: km 43+610- km 94+000,INFRAESTRUCTURA,ESTRUCTURAS FISICAS,52.3,KM,"52,3",2.934237e+08,2019-03,2.559166e+08,NaN,NaN,EJECUCION,both
4,2005,R,GOBIERNOS REGIONALES,99,GOBIERNOS REGIONALES,460,GOBIERNO REGIONAL TACNA,931,1,REGION TACNA-SEDE CENTRAL,23,TACNA,1,TACNA,1,REGION TACNA-SEDE CENTRAL,2,PROYECTO,2015918,PROYECTO VIAL TACNA LA PAZ,16,TRANSPORTE,23,TACNA,1,RECURSOS ORDINARIOS,15,FONDO DE COMPENSACION REGIONAL,0.0,97546.66,0.0,0,0,0.0,97546.66,2015918.0,"RED VIAL INTEGRACION VIAL TACNA- LA PAZ, TRAMO...",Tramo I: km 43+610- km 94+000,INFRAESTRUCTURA,ESTRUCTURAS FISICAS,52.3,KM,"52,3",2.934237e+08,2019-02,2.472077e+08,NaN,NaN,EJECUCION,both


### <font color=green> 11. REVISAR EL RESULTADO DE LA INTEGRACIÓN </font>

In [16]:
# La columna _merge indica de dónde viene cada registro.

print("Resultado de la integración:")
display(base_integrada["_merge"].value_counts())

print("Valores faltantes en la base integrada:")
display(base_integrada.isnull().sum())

print("Resumen de la base integrada:")
display(base_integrada.describe(include="all"))

Resultado de la integración:


_merge
both          12186964
left_only       524199
right_only           0
Name: count, dtype: int64

Valores faltantes en la base integrada:


ANO_EJE                                     0
NIVEL_GOBIERNO                              0
NIVEL_GOBIERNO_NOMBRE                       0
SECTOR                                      0
SECTOR_NOMBRE                               0
PLIEGO                                      0
PLIEGO_NOMBRE                               0
SEC_EJEC                                    0
EJECUTORA                                   0
EJECUTORA_NOMBRE                            0
DEPARTAMENTO_EJECUTORA                      0
DEPARTAMENTO_EJECUTORA_NOMBRE               0
PROVINCIA_EJECUTORA                         0
PROVINCIA_EJECUTORA_NOMBRE                  0
DISTRITO_EJECUTORA                          0
DISTRITO_EJECUTORA_NOMBRE                   0
TIPO_ACT_PROY                               0
TIPO_ACT_PROY_NOMBRE                        0
PRODUCTO_PROYECTO                           0
PRODUCTO_PROYECTO_NOMBRE                    0
FUNCION                                     0
FUNCION_NOMBRE                    

Resumen de la base integrada:


,ANO_EJE,NIVEL_GOBIERNO,NIVEL_GOBIERNO_NOMBRE,SECTOR,SECTOR_NOMBRE,PLIEGO,PLIEGO_NOMBRE,SEC_EJEC,EJECUTORA,EJECUTORA_NOMBRE,DEPARTAMENTO_EJECUTORA,DEPARTAMENTO_EJECUTORA_NOMBRE,PROVINCIA_EJECUTORA,PROVINCIA_EJECUTORA_NOMBRE,DISTRITO_EJECUTORA,DISTRITO_EJECUTORA_NOMBRE,TIPO_ACT_PROY,TIPO_ACT_PROY_NOMBRE,PRODUCTO_PROYECTO,PRODUCTO_PROYECTO_NOMBRE,FUNCION,FUNCION_NOMBRE,DEPARTAMENTO_META,DEPARTAMENTO_META_NOMBRE,FUENTE_FINANCIAMIENTO,FUENTE_FINANCIAMIENTO_NOMBRE,RUBRO,RUBRO_NOMBRE,COSTO_ACTUAL,MONTO_EJECUCION_HASTA_HACE_2_ANOS,MONTO_EJECUCION_ANO_ANTERIOR,MONTO_PIA,MONTO_PIM,MONTO_DEVENGADO_ANO_EJE,MONTO_EJECUCION_TOTAL,CODIGO_UNICO,DES_PRODUCTO,DES_ACCION,DES_TIPO_COMPONENTE,DES_UM_PRODU,VAL_META_PRODU,DES_UM_CAPAC,VAL_META_CAPAC,COSTO_INVERSION,PERIODO,VALORIZ_ACUM,AVANCE,DES_OBSERVACIONES,DES_ETAPA,_merge
count,1.271116e+07,12711163,12711163,12711163,12711163,12711163,12711163,1.271116e+07,1.271116e+07,12711163,1.271116e+07,12711163,1.271116e+07,12711163,1.271116e+07,12711163,12711163.0,12711163,1.271116e+07,12711163,1.271116e+07,12711163,1.271116e+07,12711163,1.271116e+07,12711163,1.271116e+07,12711163,1.271116e+07,1.271116e+07,1.271116e+07,1.271116e+07,1.271116e+07,1.271116e+07,1.271116e+07,1.218696e+07,12154034,10879988,10993203,9058003,9.558182e+06,5763237,8858447.0,1.217677e+07,9091723,3.919348e+06,1.966040e+06,4621109,12186964,12711163
unique,NaN,3,3,61,33,331,233,NaN,NaN,2520,NaN,26,NaN,197,NaN,2521,NaN,1,NaN,53314,NaN,33,NaN,27,NaN,5,NaN,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54809,32726,9,57,NaN,43,108535.0,NaN,635,NaN,NaN,181903,7,2
top,NaN,M,GOBIERNOS LOCALES,,,,,NaN,NaN,REGION UCAYALI-SEDE CENTRAL,NaN,LIMA,NaN,LIMA,NaN,REGION UCAYALI-SEDE CENTRAL,NaN,PROYECTO,NaN,ESTUDIOS DE PRE-INVERSION,NaN,SANEAMIENTO,NaN,CUSCO,NaN,RECURSOS DETERMINADOS,NaN,"CANON Y SOBRECANON, REGALIAS, RENTA DE ADUANAS...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INFRAESTRUCTURA,Inventario físico Covid - 19,INFRAESTRUCTURA,NÚMERO DE ESTRUCTURAS FÍSICAS,NaN,M2,1.0,NaN,2021-01,NaN,NaN,MITIGACION AMBIENTAL,Ejecución física (C),both
freq,NaN,6409925,6409925,6409925,6409925,6409925,6409925,NaN,NaN,605402,NaN,2493273,NaN,2297056,NaN,605402,NaN,12711163,NaN,82857,NaN,2567365,NaN,1894866,NaN,6900084,NaN,4831418,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1188949,944384,7332954,2240020,NaN,3030321,1006097.0,NaN,140750,NaN,NaN,14344,2930579,12186964
mean,2.021687e+03,NaN,NaN,NaN,NaN,NaN,NaN,1.523438e+05,5.359509e+04,NaN,1.213782e+01,NaN,3.343032e+00,NaN,6.459906e+00,NaN,2.0,NaN,2.349901e+06,NaN,1.577508e+01,NaN,1.183442e+01,NaN,3.514954e+00,NaN,1.127818e+01,NaN,5.822376e+07,1.772121e+06,3.327798e+05,2.894934e+05,3.571192e+05,1.832243e+05,2.288125e+06,2.362252e+06,NaN,NaN,NaN,NaN,1.587974e+05,NaN,NaN,1.007539e+07,NaN,1.152088e+07,2.992304e+09,NaN,NaN,NaN
std,4.104899e+00,NaN,NaN,NaN,NaN,NaN,NaN,1.500011e+05,7.022531e+04,NaN,6.712707e+00,NaN,3.458105e+00,NaN,7.878047e+00,NaN,0.0,NaN,1.803827e+05,NaN,5.619605e+00,NaN,7.108545e+00,NaN,1.758183e+00,NaN,7.842820e+00,NaN,3.310905e+08,1.123128e+07,6.465950e+06,4.983883e+06,4.986682e+06,3.247852e+06,1.328714e+07,1.698127e+05,NaN,NaN,NaN,NaN,9.753634e+06,NaN,NaN,1.414361e+08,NaN,1.835389e+08,5.509105e+11,NaN,NaN,NaN
min,2.005000e+03,NaN,NaN,NaN,NaN,NaN,NaN,3.000000e+00,1.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,NaN,2.0,NaN,2.000028e+06,NaN,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,-3.216010e+05,-7.873040e+05,0.000000e+00,0.000000e+00,0.000000e+00,-7.873040e+05,2.000846e+06,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,-1.041259e+07,NaN,1.000000e-02,0.000000e+00,NaN,NaN,NaN
25%,2.020000e+03,NaN,NaN,NaN,NaN,NaN,NaN,9.310000e+02,1.000000e+00,NaN,6.000000e+00,NaN,1.000000e+00,NaN,1.000000e+00,NaN,2.0,NaN,2.201029e+06,NaN,1.000000e+01,NaN,6.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,3.257637e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.215000e+03,2.229569e+06,NaN,NaN,NaN,NaN,1.000000e+00,NaN,NaN,2.000000e+04,NaN,1.373135e+05,2.529000e+01,NaN,NaN,NaN
50%,2.02300

### <font color=green> 12. PLANTILLA SIMPLE DE DICCIONARIO DE DATOS </font>

In [17]:
import pandas as pd

# Plantilla completa del diccionario de datos (MEF y mef_2)
diccionario_data = [
    # --- VARIABLES DE LA FUENTE 1 (MEF) ---
    ['ANO_EJE', 'mef_seguimiento_financiero', "Año de ejecución del presupuesto.", 'Numérica', 'Intervalo', 'Discreta (Fecha)', ''],
    ['NIVEL_GOBIERNO', 'mef_seguimiento_financiero', "Código que identifica el Nivel de Gobierno: E, R, M.", 'Categórica', 'Nominal', 'Código agrupador', ''],
    ['NIVEL_GOBIERNO_NOMBRE', 'mef_seguimiento_financiero', "Descripción de Nivel de Gobierno: Nacional, Regionales, Locales.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['SECTOR', 'mef_seguimiento_financiero', "Código de Sector al que pertenece la Entidad.", 'Categórica', 'Nominal', 'Código agrupador', ''],
    ['SECTOR_NOMBRE', 'mef_seguimiento_financiero', "Descripción de código del Sector al que pertenece la Entidad.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['PLIEGO', 'mef_seguimiento_financiero', "Código de Pliego al que pertenece la Entidad.", 'Categórica', 'Nominal', 'Código agrupador', ''],
    ['PLIEGO_NOMBRE', 'mef_seguimiento_financiero', "Descripción de código del Pliego al que pertenece la Entidad.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['SEC_EJEC', 'mef_seguimiento_financiero', "Código que identifica a una Entidad.", 'Categórica', 'Nominal', 'Identificador de entidad', ''],
    ['EJECUTORA', 'mef_seguimiento_financiero', "Código institucional que identifica a una Entidad.", 'Categórica', 'Nominal', 'Identificador de entidad', ''],
    ['EJECUTORA_NOMBRE', 'mef_seguimiento_financiero', "Nombre de la Entidad.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['DEPARTAMENTO_EJECUTORA', 'mef_seguimiento_financiero', "Código de departamento donde se ubica la Entidad.", 'Categórica', 'Nominal', 'Código geográfico', ''],
    ['DEPARTAMENTO_EJECUTORA_NOMBRE', 'mef_seguimiento_financiero', "Nombre de departamento donde se ubica la Entidad.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['PROVINCIA_EJECUTORA', 'mef_seguimiento_financiero', "Código de provincia donde se ubica la Entidad.", 'Categórica', 'Nominal', 'Código geográfico', ''],
    ['PROVINCIA_EJECUTORA_NOMBRE', 'mef_seguimiento_financiero', "Nombre de provincia donde se ubica la Entidad.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['DISTRITO_EJECUTORA', 'mef_seguimiento_financiero', "Código de distrito donde se ubica la Entidad.", 'Categórica', 'Nominal', 'Código geográfico', ''],
    ['DISTRITO_EJECUTORA_NOMBRE', 'mef_seguimiento_financiero', "Nombre de distrito donde se ubica la Entidad.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['TIPO_ACT_PROY', 'mef_seguimiento_financiero', "Código que muestra si es proyecto o producto.", 'Categórica', 'Nominal', 'Código agrupador', ''],
    ['TIPO_ACT_PROY_NOMBRE', 'mef_seguimiento_financiero', "Nombre que muestra si es proyecto o producto.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['PRODUCTO_PROYECTO', 'mef_seguimiento_financiero', "Código que identifica a un proyecto o producto.", 'Categórica', 'Nominal', 'Identificador (Llave primaria)', 'Usar como llave para Join (CUI)'],
    ['PRODUCTO_PROYECTO_NOMBRE', 'mef_seguimiento_financiero', "Descripción de un proyecto o producto.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['FUNCION', 'mef_seguimiento_financiero', "Código de Función.", 'Categórica', 'Nominal', 'Código agrupador', ''],
    ['FUNCION_NOMBRE', 'mef_seguimiento_financiero', "Corresponde al nivel máximo de agregación de acciones.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['DEPARTAMENTO_META', 'mef_seguimiento_financiero', "Código de Departamento de la meta.", 'Categórica', 'Nominal', 'Código geográfico', ''],
    ['DEPARTAMENTO_META_NOMBRE', 'mef_seguimiento_financiero', "Nombre del departamento de la meta.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['FUENTE_FINANCIAMIENTO', 'mef_seguimiento_financiero', "Código de la Fuente de Financiamiento.", 'Categórica', 'Nominal', 'Código agrupador', ''],
    ['FUENTE_FINANCIAMIENTO_NOMBRE', 'mef_seguimiento_financiero', "Descripción de la Fuente de Financiamiento.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['RUBRO', 'mef_seguimiento_financiero', "Código del Rubro que puede utilizar la Entidad.", 'Categórica', 'Nominal', 'Código agrupador', ''],
    ['RUBRO_NOMBRE', 'mef_seguimiento_financiero', "Descripción del Rubro.", 'Texto', 'Nominal', 'Texto descriptivo', ''],
    ['COSTO_ACTUAL', 'mef_seguimiento_financiero', "Costo actual del proyecto.", 'Numérica', 'Razón', 'Continua', 'Posiblemente sucia (suele venir en 0)'],
    ['MONTO_EJECUCION_HASTA_HACE_2_ANOS', 'mef_seguimiento_financiero', "Monto acumulado histórico (hasta hace 2 años).", 'Numérica', 'Razón', 'Continua', ''],
    ['MONTO_EJECUCION_ANO_ANTERIOR', 'mef_seguimiento_financiero', "Monto de la ejecución del año anterior.", 'Numérica', 'Razón', 'Continua', ''],
    ['MONTO_PIA', 'mef_seguimiento_financiero', "Presupuesto Institucional de Apertura.", 'Numérica', 'Razón', 'Continua', ''],
    ['MONTO_PIM', 'mef_seguimiento_financiero', "Presupuesto Institucional Modificado.", 'Numérica', 'Razón', 'Continua', ''],
    ['MONTO_DEVENGADO_ANO_EJE', 'mef_seguimiento_financiero', "Monto ejecutado anual a la fecha.", 'Numérica', 'Razón', 'Continua', ''],
    ['MONTO_EJECUCION_TOTAL', 'mef_seguimiento_financiero', "Monto Total de la ejecución acumulada.", 'Numérica', 'Razón', 'Continua', 'Variable Target/KPI financiera'],

    # --- VARIABLES DE LA FUENTE 2 (mef_2) ---
    ['CODIGO_UNICO', 'mef_2_avance_fisico', "Codigo unico de la inversión (CUI).", 'Categórica', 'Nominal', 'Identificador (Llave primaria)', 'Usar como llave para Join (CUI)'],
    ['DES_PRODUCTO', 'mef_2_avance_fisico', "Descripción de Producto", 'Texto', 'Nominal', 'Texto libre/descriptivo', ''],
    ['DES_ACCION', 'mef_2_avance_fisico', "Descripción de Acciones", 'Texto', 'Nominal', 'Texto libre/descriptivo', 'Genera múltiples filas por CUI'],
    ['DES_TIPO_COMPONENTE', 'mef_2_avance_fisico', "Descripcion de Tipo de Componente o Item", 'Categórica', 'Nominal', 'Atributo de clasificación', ''],
    ['DES_UM_PRODU', 'mef_2_avance_fisico', "Descripción de unidad de medida de producto", 'Categórica', 'Nominal', 'Atributo de clasificación', ''],
    ['VAL_META_PRODU', 'mef_2_avance_fisico', "Valor meta de producto", 'Numérica', 'Razón', 'Continua', ''],
    ['DES_UM_CAPAC', 'mef_2_avance_fisico', "Descripción de unidad de medida de capacidad", 'Categórica', 'Nominal', 'Atributo de clasificación', ''],
    ['VAL_META_CAPAC', 'mef_2_avance_fisico', "Valor meta de capacidad", 'Numérica', 'Razón', 'Continua', 'Pandas la detectó como texto (sucia)'],
    ['COSTO_INVERSION', 'mef_2_avance_fisico', "Costo de inversión real (según ET)", 'Numérica', 'Razón', 'Continua', 'Sustituye al COSTO_ACTUAL del MEF'],
    ['PERIODO', 'mef_2_avance_fisico', "Indica el periodo del gasto", 'Numérica', 'Intervalo', 'Fecha', ''],
    ['VALORIZ_ACUM', 'mef_2_avance_fisico', "Monto acumulado de valorización física", 'Numérica', 'Razón', 'Continua', 'Alta cantidad de valores nulos'],
    ['AVANCE', 'mef_2_avance_fisico', "Indica el porcentaje de avance fisico acumulado", 'Numérica', 'Razón', 'Continua', 'KPI Clave (alto porcentaje de nulos)'],
    ['DES_OBSERVACIONES', 'mef_2_avance_fisico', "Comentarios adicionales de la obra", 'Texto', 'Nominal', 'Texto libre', ''],
    ['DES_ETAPA', 'mef_2_avance_fisico', "Etapa de la ejecución (FYE, Expediente, Ejecución)", 'Categórica', 'Ordinal', 'Fase de ejecución', 'Posee un orden cronológico lógico']
]

# Crear el DataFrame final
columnas = ['variable', 'fuente', 'descripcion', 'tipo_general', 'escala', 'rol_o_subtipo', 'observaciones']
diccionario_datos_final = pd.DataFrame(diccionario_data, columns=columnas)

# Mostrar el resultado en el cuaderno
print("Diccionario de Datos generado con éxito:")
display(diccionario_datos_final)

Diccionario de Datos generado con éxito:


,variable,fuente,descripcion,tipo_general,escala,rol_o_subtipo,observaciones
0,ANO_EJE,mef_seguimiento_financiero,Año de ejecución del presupuesto.,Numérica,Intervalo,Discreta (Fecha),
1,NIVEL_GOBIERNO,mef_seguimiento_financiero,"Código que identifica el Nivel de Gobierno: E,...",Categórica,Nominal,Código agrupador,
2,NIVEL_GOBIERNO_NOMBRE,mef_seguimiento_financiero,"Descripción de Nivel de Gobierno: Nacional, Re...",Texto,Nominal,Texto descriptivo,
3,SECTOR,mef_seguimiento_financiero,Código de Sector al que pertenece la Entidad.,Categórica,Nominal,Código agrupador,
4,SECTOR_NOMBRE,mef_seguimiento_financiero,Descripción de código del Sector al que perten...,Texto,Nominal,Texto descriptivo,
5,PLIEGO,mef_seguimiento_financiero,Código de Pliego al que pertenece la Entidad.,Categórica,Nominal,Código agrupador,
6,PLIEGO_NOMBRE,mef_seguimiento_financiero,Descripción de código del Pliego al que perten...,Texto,Nominal,Texto descriptivo,
7,SEC_EJEC,mef_seguimiento_financiero,Código que identifica a una Entidad.,Categórica,Nominal,Identificador de entidad,
8,EJECUTORA,mef_seguimiento_financiero,Código institucional que identifica a una Enti...,Categórica,Nominal,Identificador de entidad,
9,EJECUTORA_NOMBRE,mef_seguimiento_financiero,Nombre de la Entidad.,Texto,Nominal,Texto descriptivo,


### <font color=green> 13. EXPORTAR RESULTADOS </font>

## Actividad final para grupos

Cada grupo debe aplicar el mismo flujo a sus propias fuentes de datos y presentar brevemente:

1. Problema que están trabajando.
2. Fuentes de datos que desean integrar.
3. Unidad de análisis.
4. Columnas comunes o posible llave de integración.
5. Tabla resumen de cada fuente.
6. Diccionario de datos inicial.
7. Problemas encontrados: valores faltantes, tipos mal detectados, categorías raras o posibles inconsistencias.

In [1]:
# EXPORTAR RESULTADOS (Solución Big Data)

# Como la base integrada supera el millón de filas (límite de Excel),
# exportaremos una muestra representativa de 100,000 filas para el Excel de la profesora,
# y guardaremos la base completa en formato CSV que no tiene límite de tamaño.

with pd.ExcelWriter("resultado_laboratorio_mef_mef_2.xlsx") as writer:
    # 1. Tabla resumen
    tabla_resumen_fuentes.to_excel(writer, sheet_name="resumen_fuentes", index=False)
    
    # 2. Base integrada (SOLO UNA MUESTRA DE 100,000 FILAS PARA QUE EXCEL NO EXPLOTE)
    base_integrada.head(100000).to_excel(writer, sheet_name="base_integrada", index=False)
    
    # 3. Diccionario
    diccionario_datos.to_excel(writer, sheet_name="diccionario_datos", index=False)

# Guardamos la verdadera base completa sin límites
base_integrada.to_csv("base_integrada_completa.csv", index=False)

print("Archivo Excel de muestra generado: resultado_laboratorio_mef_mef_2.xlsx")
print("Archivo CSV completo generado: base_integrada_completa.csv")

NameError: name 'pd' is not defined

###  ANÁLISIS DE CALIDAD DE DATOS (DATA QUALITY) E INCONSISTENCIAS

De acuerdo con el marco teórico, al combinar fuentes que describen las mismas entidades suelen aparecer conflictos que deben resolverse antes del análisis[cite: 2]. En nuestra integración (MEF + mef_2) hemos detectado los siguientes problemas:

**1. Datos Faltantes:**
* *Definición:* Ocurre cuando un dato está presente en una fuente, pero ausente en otra[cite: 2]. En muchos casos, los datos inconsistentes se tratan provisionalmente como valores faltantes hasta resolver la discrepancia[cite: 2].
* *Hallazgo:* La variable `COSTO_ACTUAL` (MEF) está completamente vacía (puros ceros). Por otro lado, nuestro KPI principal, la variable `AVANCE` (mef_2), presenta casi un 90% de valores nulos, requiriendo estrategias de imputación o filtrado estricto.

**2. Datos Duplicados:**
* *Definición:* Se presenta cuando un registro aparece más de una vez (por ejemplo, una misma transacción duplicada)[cite: 2].
* *Hallazgo (Riesgo Crítico):* La llave `CODIGO_UNICO` en el mef_2 no es única por proyecto; se repite porque registra historiales mensuales y múltiples "Acciones" de obra. Al realizar el cruce sin tratamiento previo, se generó una explosión de registros (casi 13 millones de filas duplicadas). **Estrategia a aplicar:** Consolidar y agrupar la base del mef_2 antes del cruce.

**3. Formatos no uniformes y Valores inválidos:**
* *Definición:* Los formatos no uniformes implican diferencias en codificaciones que obligan a unificar formatos[cite: 2]. Los valores inválidos se dan cuando una variable registra un dato que no admite[cite: 2].
* *Hallazgo:* Detectamos un problema de codificación de formato (carácter BOM `ï»¿`) en la cabecera del mef_2. Además, encontramos valores inválidos (caracteres no numéricos) en la columna `VAL_META_CAPAC` del mef_2, lo que provocó que Pandas la clasificara erróneamente como texto (`object`).

**4. Datos Redundantes:**
* *Definición:* Se da cuando dos atributos contienen la misma información en distintas fuentes[cite: 2].
* *Hallazgo:* Las variables `COSTO_ACTUAL` (MEF) y `COSTO_INVERSION` (mef_2) son redundantes al medir el presupuesto de la obra. Optaremos por priorizar la variable del mef_2.